##### 1. Import Numpy and Pandas Setup

In [1]:
# -------------------------------------------------
# 1. Import Libraries
# -------------------------------------------------

import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime

In [2]:
# -------------------------------------------------
# 2. Load CSV and Clean to Base Financial Data Only
# -------------------------------------------------

df = pd.read_csv("../data/processed/elv_financials_validated_2026-04-19_15-45.csv")

# -------------------------------------------------
# 3. Dataset Structural Summary
# -------------------------------------------------

# Ensure year is numeric
df["year"] = pd.to_numeric(df["year"], errors="coerce")

latest_year = df["year"].max()
first_year = df["year"].min()
years_available = sorted(df["year"].dropna().unique())

# ---- Column Table ----
column_table = (
    pd.DataFrame({
        "column_name": sorted(df.columns)
    })
    .reset_index()
    .rename(columns={"index": "column_number"})
)

# ---- Year Table ----
year_table = pd.DataFrame({
    "available_years": years_available
})

# ---- Display ----
print("\n=== DATASET STRUCTURE SUMMARY ===")
print(f"Total Columns: {len(df.columns)}")
print(f"Total Rows: {len(df)}")
print(f"First Year: {first_year}")
print(f"Latest Year: {latest_year}")
print(f"Number of Years Available: {len(years_available)}")

print("\n--- Available Years ---")
display(year_table)

print("\n--- Column Schema ---")
display(column_table)


=== DATASET STRUCTURE SUMMARY ===
Total Columns: 54
Total Rows: 40
First Year: 2021
Latest Year: 2024
Number of Years Available: 4

--- Available Years ---


,available_years
0,2021
1,2022
2,2023
3,2024



--- Column Schema ---


,column_number,column_name
0,0,accounts_payable
1,1,accounts_receivable
2,2,asset_turnover
3,3,cash_liquidity
4,4,cash_register_and_bank
5,5,company
6,6,current_assets
7,7,current_liabilities
8,8,debt_ratio
9,9,ebitda


In [3]:
# -------------------------------------------------
# 4. List Available Target Companies
# -------------------------------------------------

def list_available_companies(df):
    """
    Displays available companies and their time coverage.
    """
    summary = (
        df.groupby("company")
          .agg(
              first_year=("year", "min"),
              last_year=("year", "max"),
              observations=("year", "count")
          )
          .sort_values("company")
          .reset_index()
    )
    print("\n=== AVAILABLE COMPANIES ===")
    display(summary)

    return summary
    
company_summary = list_available_companies(df)


=== AVAILABLE COMPANIES ===


,company,first_year,last_year,observations
0,Arboga Bilskrot AB,2021,2024,4
1,Autocirc Svensk Bilåtervinning AB,2021,2024,4
2,Bilåtervinning i Tibro AB,2021,2024,4
3,Bjuhrs Bil AB,2021,2024,4
4,Ekholms Bildemontering AB,2021,2024,4
5,Köpings Bildemontering AB,2021,2024,4
6,Mariestads Bildemontering och Återvinningstekn...,2021,2024,4
7,Nya Högåsens Bildemontering AB,2021,2024,4
8,Vingåkers Bilskrot AB,2021,2024,4
9,Örebro Bildemontering AB,2021,2024,4


In [4]:
# -------------------------------------------------
# 5. Select Target Company
# -------------------------------------------------

target = "Ekholms Bildemontering AB"

df_target = df[df["company"] == target].copy()

if df_target.empty:
    raise ValueError(f"Target '{target}' not found in dataset.")

latest_year = df_target["year"].max()
years_available = sorted(df_target["year"].unique())

print("\n=== TARGET SELECTION SUMMARY ===")
print(f"Selected Target: {target}")
print(f"Latest Year Available: {latest_year}")
print(f"Years Available: {years_available}")
print(f"Number of Observations: {len(df_target)}")

# -------------------------------------------------
# Preview First 3 Rows (Full Column Values)
# -------------------------------------------------

display(
    df_target.head(5)
        .style
        .set_caption("Target Data Preview (First 5 Rows)")
        .format(precision=2)
)


=== TARGET SELECTION SUMMARY ===
Selected Target: Ekholms Bildemontering AB
Latest Year Available: 2024
Years Available: [np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]
Number of Observations: 4


,company,year,accounts_payable,accounts_receivable,cash_register_and_bank,current_assets,current_liabilities,ebitda,employees,equity,financial_costs,financial_fixed_assets,financial_income,fixed_assets,free_equity,intangible_fixed_assets,inventory,long-term_liabilities,net_sales,operating_expenses,operating_profit_after_depreciation,other_turnover,personnel_cost_per_employee,personnel_costs_per_employee_(in_1000),profit_before_tax,provisions,result_after_net_financial_items,results_for_the_year,stock_change,tangible_fixed_assets,tax_on_the_year's_profit,total_assets,total_equity_and_liabilities,turnover,untaxed_reserves,operating_margin,net_margin,liquidity_ratio,operating_expense_ratio,financial_cost_ratio,market_share,hhi_contribution,hhi,effective_firms,profit_share,profit_margin,ebitda_margin,equity_ratio,debt_ratio,return_on_equity,return_on_total_capital,cash_liquidity,asset_turnover,yoy_revenue_growth
16,Ekholms Bildemontering AB,2021,79.00,58.00,1688.00,2188.00,1222.00,858.00,4.00,819.00,-15.00,60.00,0.00,80.00,699.00,0.00,272.00,0.00,6169.00,-5577.00,851.00,259.00,nan,662.00,610.00,0.00,837.00,477.00,0.00,20.00,-133.00,2267.00,2267.00,6428.00,227.00,13.24,0.14,1.79,-0.90,-0.00,0.04,0.00,0.30,3.39,0.03,7.42,13.35,0.36,0.64,58.24,37.54,1.43,2.84,nan
17,Ekholms Bildemontering AB,2022,110.00,61.00,2207.00,2432.00,1203.00,1153.00,4.00,792.00,-12.00,60.00,0.00,72.00,672.00,0.00,0.00,0.00,6256.00,-5373.00,1146.00,263.00,nan,692.00,852.00,0.00,1134.00,671.00,0.00,12.00,-181.00,2505.00,2505.00,6519.00,509.00,17.58,0.18,2.02,-0.86,-0.00,0.04,0.00,0.31,3.20,0.06,10.29,17.69,0.32,0.68,84.72,45.75,1.89,2.60,1.42
18,Ekholms Bildemontering AB,2023,102.00,109.00,589.00,1309.00,679.00,161.00,4.00,203.00,-7.00,60.00,5.00,111.00,83.00,0.00,0.00,0.00,3671.00,-4064.00,142.00,535.00,nan,584.00,112.00,0.00,139.00,82.00,0.00,51.00,-30.00,1420.00,1420.00,4206.00,537.00,3.38,0.04,1.93,-1.11,-0.00,0.02,0.00,0.36,2.81,0.01,1.95,3.83,0.14,0.86,40.39,10.00,1.03,2.96,-35.48
19,Ekholms Bildemontering AB,2024,92.00,147.00,629.00,1049.00,587.00,-95.00,4.00,160.00,-26.00,60.00,19.00,95.00,40.00,0.00,0.00,0.00,3214.00,-3561.00,-112.00,233.00,nan,485.00,20.00,0.00,-120.00,7.00,0.00,35.00,-13.00,1144.00,1144.00,3447.00,397.00,-3.25,-0.04,1.79,-1.11,-0.01,0.02,0.00,0.37,2.71,-0.00,0.20,-2.76,0.14,0.86,4.38,-9.79,1.32,3.01,-18.05


In [6]:
# -------------------------------------------------
# Quintile Classification Engine
# -------------------------------------------------

def classify_quintile(
    target_value,
    industry_series,
    higher_is_better=True
):
    """
    Classify target into quintile (1–5 scale).
    Returns dict with numeric quintile and label.
    """

    if pd.isna(target_value):
        return {"quintile": np.nan, "label": "Undefined"}

    clean = industry_series.dropna()

    if len(clean) < 5:
        return {"quintile": np.nan, "label": "Insufficient Data"}

    # Percentile position
    percentile = (clean < target_value).sum() / len(clean)

    # Invert if lower is better
    if not higher_is_better:
        percentile = 1 - percentile

    quintile = int(np.ceil(percentile * 5))

    labels = {
        1: "Bottom 20%",
        2: "20–40%",
        3: "40–60%",
        4: "60–80%",
        5: "Top 20%"
    }

    return {
        "quintile": quintile,
        "label": labels.get(quintile, "Undefined")
    }

##### 4. Revenue Analysis

In [7]:
# -------------------------------------------------
# 1. Inspect Target Revenue (Latest Year)
# -------------------------------------------------

print("\n--- TARGET DATA (Latest Year) ---")

target_rev_df = df_target[df_target["year"] == latest_year][
    ["company", "year", "turnover"]
]

display(target_rev_df)


# -------------------------------------------------
# 2. Inspect Industry Revenue (Latest Year)
# -------------------------------------------------

print("\n--- INDUSTRY DATA (Latest Year) ---")

industry_rev_df = df[df["year"] == latest_year][
    ["company", "year", "turnover"]
]

display(industry_rev_df)

industry_sample_size = len(industry_rev_df)
print("\nIndustry sample size:", industry_sample_size)


# -------------------------------------------------
# 3. Extract Inputs (Safe)
# -------------------------------------------------

if len(target_rev_df) == 0:
    rev_target_latest = np.nan
else:
    rev_target_latest = target_rev_df["turnover"].iloc[0]

industry_rev_series = industry_rev_df["turnover"]
rev_industry_mean = industry_rev_series.mean()

print("\n--- CALCULATION INPUTS ---")
print(f"Target Revenue: {rev_target_latest:,.2f}" if pd.notna(rev_target_latest) else "Target Revenue: NaN")
print(f"Industry Mean Revenue: {rev_industry_mean:,.2f}")


# -------------------------------------------------
# 4. Relative Scale Insight (Keep Economic Meaning)
# -------------------------------------------------

if pd.notna(rev_target_latest) and rev_industry_mean != 0:
    relative_pct = (rev_target_latest / rev_industry_mean - 1) * 100
else:
    relative_pct = np.nan

print("\n--- RELATIVE SCALE ---")
print("Relative % = (Target / Industry Mean - 1) * 100")
print(f"= {relative_pct:.2f}%" if pd.notna(relative_pct) else "= NaN")


# -------------------------------------------------
# 5. Quintile Classification (Distribution-Based)
# -------------------------------------------------

rev_quintile_result = classify_quintile(
    rev_target_latest,
    industry_rev_series,
    higher_is_better=True
)

rev_snapshot = {
    "target": rev_target_latest,
    "industry_mean": rev_industry_mean,
    "relative_%": relative_pct,
    "quintile": rev_quintile_result["quintile"],
    "classification": rev_quintile_result["label"]
}

print("\n--- REVENUE SNAPSHOT (QUINTILE-BASED) ---")
display(pd.DataFrame([rev_snapshot]))


--- TARGET DATA (Latest Year) ---


,company,year,turnover
19,Ekholms Bildemontering AB,2024,3447.0



--- INDUSTRY DATA (Latest Year) ---


,company,year,turnover
3,Arboga Bilskrot AB,2024,4813.0
7,Autocirc Svensk Bilåtervinning AB,2024,41596.0
11,Bilåtervinning i Tibro AB,2024,8328.0
15,Bjuhrs Bil AB,2024,8123.0
19,Ekholms Bildemontering AB,2024,3447.0
23,Köpings Bildemontering AB,2024,7825.0
27,Mariestads Bildemontering och Återvinningstekn...,2024,7953.0
31,Nya Högåsens Bildemontering AB,2024,1857.0
35,Vingåkers Bilskrot AB,2024,4246.0
39,Örebro Bildemontering AB,2024,112400.0



Industry sample size: 10

--- CALCULATION INPUTS ---
Target Revenue: 3,447.00
Industry Mean Revenue: 20,058.80

--- RELATIVE SCALE ---
Relative % = (Target / Industry Mean - 1) * 100
= -82.82%

--- REVENUE SNAPSHOT (QUINTILE-BASED) ---


,target,industry_mean,relative_%,quintile,classification
0,3447.0,20058.8,-82.815522,1,Bottom 20%


##### 5. Profit Margin Analysis

In [ ]:
# -------------------------------------------------
# 1. Inspect Target Data (Latest Year)
# -------------------------------------------------

print("\n--- TARGET DATA (Latest Year) ---")

target_pm_df = df_target[df_target["year"] == latest_year][
    ["company", "year", "results_for_the_year", "turnover", "profit_margin"]
]

display(target_pm_df)


# -------------------------------------------------
# 2. Inspect Industry Data (Latest Year)
# -------------------------------------------------

print("\n--- INDUSTRY DATA (Latest Year) ---")

industry_pm_df = df[df["year"] == latest_year][
    ["company", "year", "results_for_the_year", "turnover", "profit_margin"]
]

display(industry_pm_df)

industry_sample_size = len(industry_pm_df)
print("\nIndustry sample size:", industry_sample_size)


# -------------------------------------------------
# 3. Extract Inputs
# -------------------------------------------------

if len(target_pm_df) == 0:
    pm_target_latest = np.nan
else:
    pm_target_latest = target_pm_df["profit_margin"].iloc[0]

industry_pm_series = industry_pm_df["profit_margin"]
pm_industry_mean = industry_pm_series.mean()

print("\n--- CALCULATION INPUTS ---")
print(
    f"Target Profit Margin: {pm_target_latest:.2f}%"
    if pd.notna(pm_target_latest)
    else "Target Profit Margin: NaN"
)
print(f"Industry Mean Profit Margin: {pm_industry_mean:.2f}%")


# -------------------------------------------------
# 4. Economic Gap (Keep for Interpretation)
# -------------------------------------------------

if pd.notna(pm_target_latest):
    pm_gap = pm_target_latest - pm_industry_mean
else:
    pm_gap = np.nan

print("\n--- GAP (Economic Insight) ---")
print("Gap (pp) = Target - Industry Mean")
print(f"= {pm_gap:.2f} pp" if pd.notna(pm_gap) else "= NaN")


# -------------------------------------------------
# 5. Quintile Classification
# -------------------------------------------------

pm_quintile_result = classify_quintile(
    pm_target_latest,
    industry_pm_series,
    higher_is_better=True   # Higher margin is better
)

pm_snapshot = {
    "target": pm_target_latest,
    "industry_mean": pm_industry_mean,
    "gap_pp": pm_gap,
    "quintile": pm_quintile_result["quintile"],
    "classification": pm_quintile_result["label"]
}

print("\n--- PROFIT MARGIN SNAPSHOT (QUINTILE-BASED) ---")
display(pd.DataFrame([pm_snapshot]))


--- TARGET DATA (Latest Year) ---


,company,year,results_for_the_year,turnover,profit_margin
19,Ekholms Bildemontering AB,2024,7.0,3447.0,0.203075



--- INDUSTRY DATA (Latest Year) ---


,company,year,results_for_the_year,turnover,profit_margin
3,Arboga Bilskrot AB,2024,78.0,4813.0,1.620611
7,Autocirc Svensk Bilåtervinning AB,2024,1011.0,41596.0,2.430522
11,Bilåtervinning i Tibro AB,2024,716.0,8328.0,8.597502
15,Bjuhrs Bil AB,2024,109.0,8123.0,1.341869
19,Ekholms Bildemontering AB,2024,7.0,3447.0,0.203075
23,Köpings Bildemontering AB,2024,642.0,7825.0,8.204473
27,Mariestads Bildemontering och Återvinningstekn...,2024,426.0,7953.0,5.356469
31,Nya Högåsens Bildemontering AB,2024,-9.0,1857.0,-0.484653
35,Vingåkers Bilskrot AB,2024,713.0,4246.0,16.792275
39,Örebro Bildemontering AB,2024,5436.0,112400.0,4.836299



Industry sample size: 10

--- CALCULATION INPUTS ---
Target Profit Margin: 0.20%
Industry Mean Profit Margin: 4.89%

--- FORMULA ---
Gap (pp) = Target - Industry Mean
= -4.69 pp

--- SNAPSHOT RESULT ---


,target,industry_mean,gap_pp,classification
0,0.203075,4.889844,-4.686769,Neutral


##### 6. Asset Turnover Analysis

In [ ]:
# -------------------------------------------------
# 1. Inspect Target Data (Latest Year)
# -------------------------------------------------

print("\n--- TARGET DATA (Latest Year) ---")

target_latest_df = df_target[df_target["year"] == latest_year][
    ["company", "year", "turnover", "total_assets", "asset_turnover"]
]

display(target_latest_df)


# -------------------------------------------------
# 2. Inspect Industry Data (Latest Year)
# -------------------------------------------------

print("\n--- INDUSTRY DATA (Latest Year) ---")

industry_latest_df = df[df["year"] == latest_year][
    ["company", "year", "turnover", "total_assets", "asset_turnover"]
]

display(industry_latest_df)

print("\nIndustry sample size:", len(industry_latest_df))


# -------------------------------------------------
# 3. Extract Inputs (Safe)
# -------------------------------------------------

if len(target_latest_df) == 0:
    at_target_latest = np.nan
else:
    at_target_latest = target_latest_df["asset_turnover"].iloc[0]

at_industry_latest = industry_latest_df["asset_turnover"].mean()

print("\n--- CALCULATION INPUTS ---")
print(
    f"Target Asset Turnover: {at_target_latest:.4f}"
    if pd.notna(at_target_latest)
    else "Target Asset Turnover: NaN"
)
print(
    f"Industry Mean Asset Turnover: {at_industry_latest:.4f}"
    if pd.notna(at_industry_latest)
    else "Industry Mean Asset Turnover: NaN"
)

# Safe gap calculation
if pd.notna(at_target_latest) and pd.notna(at_industry_latest):
    at_gap = at_target_latest - at_industry_latest
else:
    at_gap = np.nan

print("\n--- FORMULA ---")
print("Gap = Target - Industry Mean")
print(f"= {at_gap:.4f}" if pd.notna(at_gap) else "= NaN")


# -------------------------------------------------
# 4. Snapshot Result (Unified Engine)
# -------------------------------------------------

at_snapshot = {
    "target": at_target_latest,
    "industry_mean": at_industry_latest,
    "gap": at_gap,
    "classification": classify_metric(
        at_target_latest,
        at_industry_latest,
        mode="ratio",        # ratio-based comparison
        threshold=0.2        # structural turnover intensity gap
    )
}

print("\n--- SNAPSHOT RESULT ---")
display(pd.DataFrame([at_snapshot]))


--- TARGET DATA (Latest Year) ---


,company,year,turnover,total_assets,asset_turnover
19,Ekholms Bildemontering AB,2024,3447.0,1144.0,3.013112



--- INDUSTRY DATA (Latest Year) ---


,company,year,turnover,total_assets,asset_turnover
3,Arboga Bilskrot AB,2024,4813.0,1573.0,3.059758
7,Autocirc Svensk Bilåtervinning AB,2024,41596.0,19928.0,2.087314
11,Bilåtervinning i Tibro AB,2024,8328.0,6604.0,1.261054
15,Bjuhrs Bil AB,2024,8123.0,4599.0,1.766254
19,Ekholms Bildemontering AB,2024,3447.0,1144.0,3.013112
23,Köpings Bildemontering AB,2024,7825.0,3668.0,2.133315
27,Mariestads Bildemontering och Återvinningstekn...,2024,7953.0,6683.0,1.190034
31,Nya Högåsens Bildemontering AB,2024,1857.0,2439.0,0.761378
35,Vingåkers Bilskrot AB,2024,4246.0,2352.0,1.805272
39,Örebro Bildemontering AB,2024,112400.0,67269.0,1.670903



Industry sample size: 10

--- CALCULATION INPUTS ---
Target Asset Turnover: 3.0131
Industry Mean Asset Turnover: 1.8748

--- FORMULA ---
Gap = Target - Industry Mean
= 1.1383

--- SNAPSHOT RESULT ---


,target,industry_mean,gap,classification
0,3.013112,1.874839,1.138272,Strong


##### 7. Return on Equity (ROE) Analysis

In [ ]:
# -------------------------------------------------
# 1. Inspect Target Data (Latest Year)
# -------------------------------------------------

print("\n--- TARGET DATA (Latest Year) ---")

target_roe_df = df_target[df_target["year"] == latest_year][
    ["company", "year", "return_on_equity"]
]

display(target_roe_df)


# -------------------------------------------------
# 2. Inspect Industry Data (Latest Year)
# -------------------------------------------------

print("\n--- INDUSTRY DATA (Latest Year) ---")

industry_roe_df = df[df["year"] == latest_year][
    ["company", "year", "return_on_equity"]
]

display(industry_roe_df)

print("\nIndustry sample size:", len(industry_roe_df))


# -------------------------------------------------
# 3. Extract Inputs (Safe)
# -------------------------------------------------

if len(target_roe_df) == 0:
    roe_target_latest = np.nan
else:
    roe_target_latest = target_roe_df["return_on_equity"].iloc[0]

roe_industry_latest = industry_roe_df["return_on_equity"].mean()

print("\n--- CALCULATION INPUTS ---")

print(
    f"Target ROE: {roe_target_latest:.2f}%"
    if pd.notna(roe_target_latest)
    else "Target ROE: NaN"
)

print(
    f"Industry Mean ROE: {roe_industry_latest:.2f}%"
    if pd.notna(roe_industry_latest)
    else "Industry Mean ROE: NaN"
)

# Safe gap calculation
if pd.notna(roe_target_latest) and pd.notna(roe_industry_latest):
    roe_gap = roe_target_latest - roe_industry_latest
else:
    roe_gap = np.nan

print("\n--- FORMULA ---")
print("Gap (pp) = Target - Industry Mean")
print(
    f"= {roe_gap:.2f} percentage points"
    if pd.notna(roe_gap)
    else "= NaN"
)


# -------------------------------------------------
# 4. Snapshot Result (Unified Engine)
# -------------------------------------------------

roe_snapshot = {
    "target": roe_target_latest,
    "industry_mean": roe_industry_latest,
    "gap_pp": roe_gap,
    "classification": classify_metric(
        roe_target_latest,
        roe_industry_latest,
        mode="absolute",   # ROE is % comparison
        threshold=5        # 5 percentage-point materiality
    )
}

print("\n--- SNAPSHOT RESULT ---")
display(pd.DataFrame([roe_snapshot]))


--- TARGET DATA (Latest Year) ---


,company,year,return_on_equity
19,Ekholms Bildemontering AB,2024,4.375



--- INDUSTRY DATA (Latest Year) ---


,company,year,return_on_equity
3,Arboga Bilskrot AB,2024,16.846652
7,Autocirc Svensk Bilåtervinning AB,2024,28.374965
11,Bilåtervinning i Tibro AB,2024,14.738576
15,Bjuhrs Bil AB,2024,3.989751
19,Ekholms Bildemontering AB,2024,4.375000
23,Köpings Bildemontering AB,2024,39.003645
27,Mariestads Bildemontering och Återvinningstekn...,2024,8.236659
31,Nya Högåsens Bildemontering AB,2024,-0.782609
35,Vingåkers Bilskrot AB,2024,66.079703
39,Örebro Bildemontering AB,2024,9.501171



Industry sample size: 10

--- CALCULATION INPUTS ---
Target ROE: 4.38%
Industry Mean ROE: 19.04%

--- FORMULA ---
Gap (pp) = Target - Industry Mean
= -14.66 percentage points

--- SNAPSHOT RESULT ---


,target,industry_mean,gap_pp,classification
0,4.375,19.036351,-14.661351,Weak


##### 8. Equity Ratio Analysis

In [ ]:
# -------------------------------------------------
# 1. Inspect Target Data (Latest Year)
# -------------------------------------------------

print("\n--- TARGET DATA (Latest Year) ---")

target_roe_df = df_target[df_target["year"] == latest_year][
    ["company", "year", "return_on_equity"]
]

display(target_roe_df)


# -------------------------------------------------
# 2. Inspect Industry Data (Latest Year)
# -------------------------------------------------

print("\n--- INDUSTRY DATA (Latest Year) ---")

industry_roe_df = df[df["year"] == latest_year][
    ["company", "year", "return_on_equity"]
]

display(industry_roe_df)

print("\nIndustry sample size:", len(industry_roe_df))


# -------------------------------------------------
# 3. Extract Inputs (Fully Safe)
# -------------------------------------------------

if len(target_roe_df) == 0:
    roe_target_latest = np.nan
else:
    roe_target_latest = target_roe_df["return_on_equity"].iloc[0]

roe_industry_latest = industry_roe_df["return_on_equity"].mean()

print("\n--- CALCULATION INPUTS ---")

print(
    f"Target ROE: {roe_target_latest:.2f}%"
    if pd.notna(roe_target_latest)
    else "Target ROE: NaN"
)

print(
    f"Industry Mean ROE: {roe_industry_latest:.2f}%"
    if pd.notna(roe_industry_latest)
    else "Industry Mean ROE: NaN"
)

# Safe gap logic
if pd.notna(roe_target_latest) and pd.notna(roe_industry_latest):
    roe_gap = roe_target_latest - roe_industry_latest
else:
    roe_gap = np.nan

print("\n--- FORMULA ---")
print("Gap (pp) = Target - Industry Mean")
print(
    f"= {roe_gap:.2f} percentage points"
    if pd.notna(roe_gap)
    else "= NaN"
)


# -------------------------------------------------
# 4. Snapshot Result (Unified Engine)
# -------------------------------------------------

roe_snapshot = {
    "target": roe_target_latest,
    "industry_mean": roe_industry_latest,
    "gap_pp": roe_gap,
    "classification": classify_metric(
        roe_target_latest,
        roe_industry_latest,
        mode="absolute",
        threshold=5
    )
}

print("\n--- SNAPSHOT RESULT ---")
display(pd.DataFrame([roe_snapshot]))


--- TARGET DATA (Latest Year) ---


,company,year,return_on_equity
19,Ekholms Bildemontering AB,2024,4.375



--- INDUSTRY DATA (Latest Year) ---


,company,year,return_on_equity
3,Arboga Bilskrot AB,2024,16.846652
7,Autocirc Svensk Bilåtervinning AB,2024,28.374965
11,Bilåtervinning i Tibro AB,2024,14.738576
15,Bjuhrs Bil AB,2024,3.989751
19,Ekholms Bildemontering AB,2024,4.375000
23,Köpings Bildemontering AB,2024,39.003645
27,Mariestads Bildemontering och Återvinningstekn...,2024,8.236659
31,Nya Högåsens Bildemontering AB,2024,-0.782609
35,Vingåkers Bilskrot AB,2024,66.079703
39,Örebro Bildemontering AB,2024,9.501171



Industry sample size: 10

--- CALCULATION INPUTS ---
Target ROE: 4.38%
Industry Mean ROE: 19.04%

--- FORMULA ---
Gap (pp) = Target - Industry Mean
= -14.66 percentage points

--- SNAPSHOT RESULT ---


,target,industry_mean,gap_pp,classification
0,4.375,19.036351,-14.661351,Weak


##### 9. Revenue Structure

In [ ]:
# -------------------------------------------------
# 1. Inspect Target Time Series
# -------------------------------------------------

print("\n--- TARGET REVENUE TIME SERIES ---")

rev_trend_df = (
    df_target[["year", "turnover"]]
        .sort_values("year")
)

display(rev_trend_df)

print("\nNumber of observations:", len(rev_trend_df))


# -------------------------------------------------
# 2. Clean Series
# -------------------------------------------------

rev_trend = (
    rev_trend_df
        .set_index("year")["turnover"]
        .dropna()
)

print("\nValid observations after NaN removal:", len(rev_trend))

rev_growth = rev_trend.pct_change()


display(pd.DataFrame({
    "Revenue": rev_trend,
    "YoY Growth": rev_growth
}))


# -------------------------------------------------
# 3. Safe Calculations
# -------------------------------------------------

if len(rev_trend) > 1 and rev_trend.iloc[0] != 0:

    periods = len(rev_trend) - 1
    start_rev = rev_trend.iloc[0]
    end_rev = rev_trend.iloc[-1]

    rev_cagr = (end_rev / start_rev) ** (1 / periods) - 1
    rev_volatility = rev_growth.std()

else:
    periods = 0
    start_rev = np.nan
    end_rev = np.nan
    rev_cagr = np.nan
    rev_volatility = np.nan


print("\n--- CALCULATION INPUTS ---")
print(f"Start Revenue: {start_rev:,.2f}" if pd.notna(start_rev) else "Start Revenue: NaN")
print(f"End Revenue: {end_rev:,.2f}" if pd.notna(end_rev) else "End Revenue: NaN")
print(f"Periods: {periods}")

print("\n--- FORMULA ---")
print("CAGR = (End / Start)^(1/Periods) - 1")

print(f"= {rev_cagr:.4f}" if pd.notna(rev_cagr) else "= NaN")

print("\nVolatility (std of YoY growth):")
print(f"= {rev_volatility:.4f}" if pd.notna(rev_volatility) else "= NaN")


# -------------------------------------------------
# 4. Structured Snapshot Output
# -------------------------------------------------

rev_structure_snapshot = {
    "initial_year": rev_trend.index.min() if len(rev_trend) > 0 else np.nan,
    "final_year": rev_trend.index.max() if len(rev_trend) > 0 else np.nan,
    "initial_revenue": start_rev,
    "final_revenue": end_rev,
    "CAGR_%": round(rev_cagr * 100, 2) if pd.notna(rev_cagr) else np.nan,
    "Volatility_%": round(rev_volatility * 100, 2) if pd.notna(rev_volatility) else np.nan,
    "observations": len(rev_trend)
}

print("\n--- REVENUE STRUCTURE SNAPSHOT ---")
display(pd.DataFrame([rev_structure_snapshot]))


--- TARGET REVENUE TIME SERIES ---


,year,turnover
16,2021,6428.0
17,2022,6519.0
18,2023,4206.0
19,2024,3447.0



Number of observations: 4

Valid observations after NaN removal: 4


,Revenue,YoY Growth
year,,
2021,6428.0,NaN
2022,6519.0,0.014157
2023,4206.0,-0.354809
2024,3447.0,-0.180456



--- CALCULATION INPUTS ---
Start Revenue: 6,428.00
End Revenue: 3,447.00
Periods: 3

--- FORMULA ---
CAGR = (End / Start)^(1/Periods) - 1
= -0.1876

Volatility (std of YoY growth):
= 0.1846

--- REVENUE STRUCTURE SNAPSHOT ---


,initial_year,final_year,initial_revenue,final_revenue,CAGR_%,Volatility_%,observations
0,2021,2024,6428.0,3447.0,-18.76,18.46,4


##### 10. Margin Structure

In [ ]:
# -------------------------------------------------
# 1. Inspect Target Margin Time Series
# -------------------------------------------------

print("\n--- TARGET PROFIT MARGIN TIME SERIES ---")

pm_trend_df = (
    df_target[["year", "profit_margin"]]
        .sort_values("year")
)

display(pm_trend_df)

print("\nObservation count:", len(pm_trend_df))


# -------------------------------------------------
# 2. Clean Series
# -------------------------------------------------

pm_trend = (
    pm_trend_df
        .set_index("year")["profit_margin"]
)

print("\n--- RAW SERIES ---")
print(pm_trend)

pm_trend_clean = pm_trend.dropna()

print("\n--- CLEAN SERIES (NaNs Removed) ---")
print(pm_trend_clean)

print("Valid observations:", len(pm_trend_clean))


# -------------------------------------------------
# 3. Safe Calculations
# -------------------------------------------------

if len(pm_trend_clean) > 0:

    pm_mean = pm_trend_clean.mean()
    pm_std = pm_trend_clean.std()
    pm_min = pm_trend_clean.min()
    pm_max = pm_trend_clean.max()

    # Endpoint change (economic drift)
    if len(pm_trend_clean) > 1:
        pm_direction = pm_trend_clean.iloc[-1] - pm_trend_clean.iloc[0]
    else:
        pm_direction = np.nan

else:
    pm_mean = pm_std = pm_min = pm_max = pm_direction = np.nan


print("\n--- FORMULAS ---")
print("Mean = Average margin")
print("StdDev = Margin volatility")
print("Min = Worst year")
print("Max = Best year")
print("Direction = Last - First (pp drift)")


# -------------------------------------------------
# 4. Margin Structure Snapshot
# -------------------------------------------------

pm_structure_snapshot = {
    "initial_year": pm_trend_clean.index.min() if len(pm_trend_clean) > 0 else np.nan,
    "final_year": pm_trend_clean.index.max() if len(pm_trend_clean) > 0 else np.nan,
    "Mean_%": round(pm_mean, 2) if pd.notna(pm_mean) else np.nan,
    "StdDev_%": round(pm_std, 2) if pd.notna(pm_std) else np.nan,
    "Min_%": round(pm_min, 2) if pd.notna(pm_min) else np.nan,
    "Max_%": round(pm_max, 2) if pd.notna(pm_max) else np.nan,
    "Direction_pp": round(pm_direction, 2) if pd.notna(pm_direction) else np.nan,
    "valid_observations": len(pm_trend_clean)
}

print("\n--- MARGIN STRUCTURE SNAPSHOT ---")
display(pd.DataFrame([pm_structure_snapshot]))


--- TARGET PROFIT MARGIN TIME SERIES ---


,year,profit_margin
16,2021,7.420660
17,2022,10.292990
18,2023,1.949596
19,2024,0.203075



Observation count: 4

--- RAW SERIES ---
year
2021     7.420660
2022    10.292990
2023     1.949596
2024     0.203075
Name: profit_margin, dtype: float64

--- CLEAN SERIES (NaNs Removed) ---
year
2021     7.420660
2022    10.292990
2023     1.949596
2024     0.203075
Name: profit_margin, dtype: float64
Valid observations: 4

--- FORMULAS ---
Mean = Average margin
StdDev = Margin volatility
Min = Worst year
Max = Best year
Direction = Last - First (pp drift)

--- MARGIN STRUCTURE SNAPSHOT ---


,initial_year,final_year,Mean_%,StdDev_%,Min_%,Max_%,Direction_pp,valid_observations
0,2021,2024,4.97,4.7,0.2,10.29,-7.22,4


##### 11. Asset Turnover Trend

In [ ]:
# -------------------------------------------------
# 1. Inspect Target Asset Turnover Time Series
# -------------------------------------------------

print("\n--- TARGET ASSET TURNOVER TIME SERIES ---")

at_trend_df = (
    df_target[["year", "asset_turnover"]]
        .sort_values("year")
)

display(at_trend_df)

print("\nObservation count:", len(at_trend_df))


# -------------------------------------------------
# 2. Clean Series
# -------------------------------------------------

at_trend = (
    at_trend_df
        .set_index("year")["asset_turnover"]
)

print("\n--- RAW SERIES ---")
print(at_trend)

at_trend_clean = at_trend.dropna()

print("\n--- CLEAN SERIES (NaNs Removed) ---")
print(at_trend_clean)

print("Valid observations:", len(at_trend_clean))


# -------------------------------------------------
# 3. Safe Calculations
# -------------------------------------------------

if len(at_trend_clean) > 0:

    at_mean = at_trend_clean.mean()
    at_std = at_trend_clean.std()

    # Direction only if ≥2 observations
    if len(at_trend_clean) > 1:
        at_direction = at_trend_clean.iloc[-1] - at_trend_clean.iloc[0]

        # Structural regression slope
        years = at_trend_clean.index.values
        values = at_trend_clean.values
        at_slope = np.polyfit(years, values, 1)[0]

    else:
        at_direction = np.nan
        at_slope = np.nan

else:
    at_mean = at_std = at_direction = at_slope = np.nan


print("\n--- FORMULAS ---")
print("Mean = Average asset turnover")
print("StdDev = Volatility")
print("Direction = Last - First")
print("Slope = Linear regression slope (structural drift)")


# -------------------------------------------------
# 4. Asset Turnover Trend Snapshot
# -------------------------------------------------

at_trend_snapshot = {
    "initial_year": at_trend_clean.index.min() if len(at_trend_clean) > 0 else np.nan,
    "final_year": at_trend_clean.index.max() if len(at_trend_clean) > 0 else np.nan,
    "Mean": round(at_mean, 4) if pd.notna(at_mean) else np.nan,
    "StdDev": round(at_std, 4) if pd.notna(at_std) else np.nan,
    "Direction": round(at_direction, 4) if pd.notna(at_direction) else np.nan,
    "Structural_Slope": round(at_slope, 4) if pd.notna(at_slope) else np.nan,
    "valid_observations": len(at_trend_clean)
}

print("\n--- ASSET TURNOVER TREND SNAPSHOT ---")
display(pd.DataFrame([at_trend_snapshot]))


--- TARGET ASSET TURNOVER TIME SERIES ---


,year,asset_turnover
16,2021,2.835465
17,2022,2.602395
18,2023,2.961972
19,2024,3.013112



Observation count: 4

--- RAW SERIES ---
year
2021    2.835465
2022    2.602395
2023    2.961972
2024    3.013112
Name: asset_turnover, dtype: float64

--- CLEAN SERIES (NaNs Removed) ---
year
2021    2.835465
2022    2.602395
2023    2.961972
2024    3.013112
Name: asset_turnover, dtype: float64
Valid observations: 4

--- FORMULAS ---
Mean = Average asset turnover
StdDev = Volatility
Direction = Last - First
Slope = Linear regression slope (structural drift)

--- ASSET TURNOVER TREND SNAPSHOT ---


,initial_year,final_year,Mean,StdDev,Direction,Structural_Slope,valid_observations
0,2021,2024,2.8532,0.1831,0.1776,0.0893,4


##### 12. ROE Trend

In [ ]:
# -------------------------------------------------
# 1. Inspect Target ROE Time Series
# -------------------------------------------------

print("\n--- TARGET ROE TIME SERIES ---")

roe_trend_df = (
    df_target[["year", "return_on_equity"]]
        .sort_values("year")
)

display(roe_trend_df)

print("\nObservation count:", len(roe_trend_df))


# -------------------------------------------------
# 2. Clean Series
# -------------------------------------------------

roe_trend = (
    roe_trend_df
        .set_index("year")["return_on_equity"]
)

print("\n--- RAW SERIES ---")
print(roe_trend)

roe_trend_clean = roe_trend.dropna()

print("\n--- CLEAN SERIES (NaNs Removed) ---")
print(roe_trend_clean)

print("Valid observations:", len(roe_trend_clean))


# -------------------------------------------------
# 3. Safe Calculations
# -------------------------------------------------

if len(roe_trend_clean) > 0:

    roe_mean = roe_trend_clean.mean()
    roe_std = roe_trend_clean.std()

    if len(roe_trend_clean) > 1:

        # Endpoint directional change
        roe_direction = roe_trend_clean.iloc[-1] - roe_trend_clean.iloc[0]

        # Structural regression slope
        years = roe_trend_clean.index.values
        values = roe_trend_clean.values
        roe_slope = np.polyfit(years, values, 1)[0]

    else:
        roe_direction = np.nan
        roe_slope = np.nan

else:
    roe_mean = roe_std = roe_direction = roe_slope = np.nan


print("\n--- FORMULAS ---")
print("Mean = Average ROE")
print("StdDev = ROE volatility")
print("Direction = Last - First")
print("Slope = Linear regression slope (structural drift)")


# -------------------------------------------------
# 4. ROE Trend Snapshot
# -------------------------------------------------

roe_trend_snapshot = {
    "initial_year": roe_trend_clean.index.min() if len(roe_trend_clean) > 0 else np.nan,
    "final_year": roe_trend_clean.index.max() if len(roe_trend_clean) > 0 else np.nan,
    "Mean_%": round(roe_mean, 2) if pd.notna(roe_mean) else np.nan,
    "StdDev_%": round(roe_std, 2) if pd.notna(roe_std) else np.nan,
    "Direction_pp": round(roe_direction, 2) if pd.notna(roe_direction) else np.nan,
    "Structural_Slope": round(roe_slope, 4) if pd.notna(roe_slope) else np.nan,
    "valid_observations": len(roe_trend_clean)
}

print("\n--- ROE TREND SNAPSHOT ---")
display(pd.DataFrame([roe_trend_snapshot]))


--- TARGET ROE TIME SERIES ---


,year,return_on_equity
16,2021,58.241758
17,2022,84.722222
18,2023,40.394089
19,2024,4.375000



Observation count: 4

--- RAW SERIES ---
year
2021    58.241758
2022    84.722222
2023    40.394089
2024     4.375000
Name: return_on_equity, dtype: float64

--- CLEAN SERIES (NaNs Removed) ---
year
2021    58.241758
2022    84.722222
2023    40.394089
2024     4.375000
Name: return_on_equity, dtype: float64
Valid observations: 4

--- FORMULAS ---
Mean = Average ROE
StdDev = ROE volatility
Direction = Last - First
Slope = Linear regression slope (structural drift)

--- ROE TREND SNAPSHOT ---


,initial_year,final_year,Mean_%,StdDev_%,Direction_pp,Structural_Slope,valid_observations
0,2021,2024,46.93,33.71,-53.87,-20.5928,4


##### 13. Equity Ratio Trend

In [ ]:
# -------------------------------------------------
# 1. Inspect Target Equity Ratio Time Series
# -------------------------------------------------

print("\n--- TARGET EQUITY RATIO TIME SERIES ---")

eq_trend_df = (
    df_target[["year", "equity_ratio"]]
        .sort_values("year")
)

display(eq_trend_df)

print("\nObservation count:", len(eq_trend_df))


# -------------------------------------------------
# 2. Clean Series
# -------------------------------------------------

eq_trend = (
    eq_trend_df
        .set_index("year")["equity_ratio"]
)

print("\n--- RAW SERIES ---")
print(eq_trend)

eq_trend_clean = eq_trend.dropna()

print("\n--- CLEAN SERIES (NaNs Removed) ---")
print(eq_trend_clean)

print("Valid observations:", len(eq_trend_clean))


# -------------------------------------------------
# 3. Safe Calculations
# -------------------------------------------------

if len(eq_trend_clean) > 0:

    eq_mean = eq_trend_clean.mean()
    eq_std = eq_trend_clean.std()

    if len(eq_trend_clean) > 1:

        # Endpoint directional change
        eq_direction = eq_trend_clean.iloc[-1] - eq_trend_clean.iloc[0]

        # Structural regression slope
        years = eq_trend_clean.index.values
        values = eq_trend_clean.values
        eq_slope = np.polyfit(years, values, 1)[0]

    else:
        eq_direction = np.nan
        eq_slope = np.nan

else:
    eq_mean = eq_std = eq_direction = eq_slope = np.nan


print("\n--- FORMULAS ---")
print("Mean = Average equity ratio")
print("StdDev = Volatility")
print("Direction = Last - First")
print("Slope = Linear regression slope (structural drift)")


# -------------------------------------------------
# 4. Equity Ratio Trend Snapshot
# -------------------------------------------------

eq_trend_snapshot = {
    "initial_year": eq_trend_clean.index.min() if len(eq_trend_clean) > 0 else np.nan,
    "final_year": eq_trend_clean.index.max() if len(eq_trend_clean) > 0 else np.nan,
    "Mean": round(eq_mean, 4) if pd.notna(eq_mean) else np.nan,
    "StdDev": round(eq_std, 4) if pd.notna(eq_std) else np.nan,
    "Direction": round(eq_direction, 4) if pd.notna(eq_direction) else np.nan,
    "Structural_Slope": round(eq_slope, 6) if pd.notna(eq_slope) else np.nan,
    "valid_observations": len(eq_trend_clean)
}

print("\n--- EQUITY RATIO TREND SNAPSHOT ---")
display(pd.DataFrame([eq_trend_snapshot]))


--- TARGET EQUITY RATIO TIME SERIES ---


,year,equity_ratio
16,2021,0.361270
17,2022,0.316168
18,2023,0.142958
19,2024,0.139860



Observation count: 4

--- RAW SERIES ---
year
2021    0.361270
2022    0.316168
2023    0.142958
2024    0.139860
Name: equity_ratio, dtype: float64

--- CLEAN SERIES (NaNs Removed) ---
year
2021    0.361270
2022    0.316168
2023    0.142958
2024    0.139860
Name: equity_ratio, dtype: float64
Valid observations: 4

--- FORMULAS ---
Mean = Average equity ratio
StdDev = Volatility
Direction = Last - First
Slope = Linear regression slope (structural drift)

--- EQUITY RATIO TREND SNAPSHOT ---


,initial_year,final_year,Mean,StdDev,Direction,Structural_Slope,valid_observations
0,2021,2024,0.2401,0.1154,-0.2214,-0.083744,4


##### 14. Equity Snapshot

In [ ]:
# -------------------------------------------------
# 1. Inspect Target Selection
# -------------------------------------------------

print("\n--- TARGET DATA (Latest Year) ---")

target_eq_df = df_target[df_target["year"] == latest_year][
    ["company", "year", "equity_ratio"]
]

display(target_eq_df)


# -------------------------------------------------
# 2. Inspect Industry Selection
# -------------------------------------------------

print("\n--- INDUSTRY DATA (Latest Year) ---")

industry_eq_df = df[df["year"] == latest_year][
    ["company", "year", "equity_ratio"]
]

display(industry_eq_df)

print("\nIndustry sample size:", len(industry_eq_df))


# -------------------------------------------------
# 3. Extract Calculation Inputs (Safe)
# -------------------------------------------------

if len(target_eq_df) == 0:
    eq_target_latest = np.nan
else:
    eq_target_latest = target_eq_df["equity_ratio"].iloc[0]

eq_industry_latest = industry_eq_df["equity_ratio"].mean()

print("\n--- CALCULATION INPUTS ---")
print(
    f"Target Equity Ratio: {eq_target_latest:.4f}"
    if pd.notna(eq_target_latest)
    else "Target Equity Ratio: NaN"
)
print(f"Industry Mean Equity Ratio: {eq_industry_latest:.4f}")


# -------------------------------------------------
# 4. Safe Gap Calculation
# -------------------------------------------------

if pd.notna(eq_target_latest):
    eq_gap = eq_target_latest - eq_industry_latest
else:
    eq_gap = np.nan

print("\n--- FORMULA ---")
print("Gap = Target - Industry Mean")
print(f"= {eq_gap:.4f}" if pd.notna(eq_gap) else "= NaN")


# -------------------------------------------------
# 5. Snapshot Result
# -------------------------------------------------

eq_snapshot = {
    "target": eq_target_latest,
    "industry_mean": eq_industry_latest,
    "gap": eq_gap,
    "classification": classify_metric(
        eq_target_latest,
        eq_industry_latest,
        mode="ratio",
        threshold=0.05   # 5 percentage point threshold
    )
}

print("\n--- EQUITY RATIO SNAPSHOT ---")
display(pd.DataFrame([eq_snapshot]))


--- TARGET DATA (Latest Year) ---


,company,year,equity_ratio
19,Ekholms Bildemontering AB,2024,0.13986



--- INDUSTRY DATA (Latest Year) ---


,company,year,equity_ratio
3,Arboga Bilskrot AB,2024,0.294342
7,Autocirc Svensk Bilåtervinning AB,2024,0.178794
11,Bilåtervinning i Tibro AB,2024,0.735615
15,Bjuhrs Bil AB,2024,0.594042
19,Ekholms Bildemontering AB,2024,0.139860
23,Köpings Bildemontering AB,2024,0.448746
27,Mariestads Bildemontering och Återvinningstekn...,2024,0.773904
31,Nya Högåsens Bildemontering AB,2024,0.471505
35,Vingåkers Bilskrot AB,2024,0.458759
39,Örebro Bildemontering AB,2024,0.850526



Industry sample size: 10

--- CALCULATION INPUTS ---
Target Equity Ratio: 0.1399
Industry Mean Equity Ratio: 0.4946

--- FORMULA ---
Gap = Target - Industry Mean
= -0.3547

--- EQUITY RATIO SNAPSHOT ---


,target,industry_mean,gap,classification
0,0.13986,0.494609,-0.354749,Weak


##### 15. Consolidate Snapshot

In [ ]:
# -------------------------------------------------
# 1. Standardize Snapshot Structure
# -------------------------------------------------

def standardize_snapshot(name, snapshot_dict, metric_type):
    return {
        "Metric": name,
        "Type": metric_type,
        "Target": snapshot_dict.get("target"),
        "Industry Mean": snapshot_dict.get("industry_mean"),
        "Gap": snapshot_dict.get("gap", snapshot_dict.get("gap_pp")),
        "Relative_%": snapshot_dict.get("relative_%"),
        "Classification": snapshot_dict.get("classification")
    }


financial_snapshot_table = pd.DataFrame([
    standardize_snapshot("Revenue", rev_snapshot, "Scale"),
    standardize_snapshot("Profit Margin", pm_snapshot, "Profitability"),
    standardize_snapshot("Asset Turnover", at_snapshot, "Efficiency"),
    standardize_snapshot("ROE", roe_snapshot, "Capital Return"),
    standardize_snapshot("Equity Ratio", eq_snapshot, "Capital Structure")
])


# -------------------------------------------------
# 2. Clean Display
# -------------------------------------------------

industry_sample_size = len(df[df["year"] == latest_year])

financial_snapshot_table = financial_snapshot_table[
    ["Metric", "Type", "Target", "Industry Mean", "Gap", "Relative_%", "Classification"]
]

print("\n=== FINANCIAL SNAPSHOT (LATEST YEAR) ===")
print(f"Target Company: {target}")
print(f"Year: {latest_year}")
print(f"Industry Sample Size: {industry_sample_size}")
print("-" * 50)

display(financial_snapshot_table)


=== FINANCIAL SNAPSHOT (LATEST YEAR) ===
Target Company: Ekholms Bildemontering AB
Year: 2024
Industry Sample Size: 10
--------------------------------------------------


,Metric,Type,Target,Industry Mean,Gap,Relative_%,Classification
0,Revenue,Scale,3447.000000,20058.800000,NaN,-82.815522,Subscale
1,Profit Margin,Profitability,0.203075,4.889844,-4.686769,NaN,Neutral
2,Asset Turnover,Efficiency,3.013112,1.874839,1.138272,NaN,Strong
3,ROE,Capital Return,4.375000,19.036351,-14.661351,NaN,Weak
4,Equity Ratio,Capital Structure,0.139860,0.494609,-0.354749,NaN,Weak
